In [ ]:
import socket

server_host = 'HYPE' # Or the IP address of HYPE
server_port = 1521

try:
    with socket.create_connection((server_host, server_port), timeout=5):
        print(f"Successfully connected to {server_host} on port {server_port}.")
except socket.error as e:
    print(f"Connection failed: {e}")
    print("Please check the SQL Server configuration and firewall settings on the server machine.")


In [ ]:
#import pyspark; print(pyspark.__version__)
!pip uninstall pyspark

In [1]:
import os, sys, findspark

# 1. Setup paths
os.environ["SPARK_HOME"] = "C:/spark"
os.environ["HADOOP_HOME"] = "C:/hadoop"
os.environ["JAVA_HOME"] = r"C:\Program Files\BellSoft\LibericaJDK-17"
#os.environ["JAVA_HOME"] = "C:/Program Files/Java/jdk-21" # Verify this path!
'''
# Force Spark to use IPv4 and standard sockets on Windows
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable
# This next line is critical for Spark 4.0 on Windows
os.environ["pyspark_server_inet_address"] = "127.0.0.1" 
'''
# 2. Initialize
findspark.init()

# 3. Initialize findspark so it uses C:/spark
findspark.init()
print("Findspark initialized successfully.") 

Findspark initialized successfully.


In [2]:
from pyspark.sql import SparkSession
import duckdb
import os
import time

# CONFIGURATION
MSSQL_JDBC_PATH = r"C:\spark_drivers\mssql-jdbc-13.2.1.jre11.jar"
DB_FILE = r"C:\duckDB\AdventureSales.duckdb"
TEMP_PARQUET = r"C:\duckDB\temp_sttwk_export"

# 1. Initialize Spark with your specific driver
spark = SparkSession.builder \
    .appName("SQLServerToDuckDB") \
    .config("spark.jars", MSSQL_JDBC_PATH) \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

print(f"The active Spark version is: {spark.version}")
print("Spark session created successfully.")



The active Spark version is: 4.0.1
Spark session created successfully.


In [4]:
# 2. Connection Properties
start_time = time.time()
url = "jdbc:sqlserver://localhost:1433;databaseName=sttwk_2025a_par;encrypt=true;trustServerCertificate=true"
#url = "jdbc:sqlserver://localhost:1433;databaseName=sales;encrypt=true;trustServerCertificate=true"
#url = "jdbc:sqlserver://localhost:1433;databaseName=discount;encrypt=true;trustServerCertificate=true"
properties = {
    "user": "sa",
    "password": "D@1ry$2014",
    "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
}

print("Step 1: Reading from SQL Server View...")

partition_sql = """
(
  SELECT *, CAST(week AS INT) as week_num 
  FROM dbo.sttwk_2022a
) as subq
"""

df = spark.read.format("jdbc") \
    .option("url", url) \
    .option("driver", "com.microsoft.sqlserver.jdbc.SQLServerDriver") \
    .option("dbtable", partition_sql) \
    .option("user", "sa") \
    .option("password", "D@1ry$2014") \
    .option("partitionColumn", "week_num") \
    .option("lowerBound", "1") \
    .option("upperBound", "53") \
    .option("numPartitions", "8") \
    .option("batchsize", "100000") \
    .load()
# Performance Tip: If 'sttwk_2022a' has a numeric ID, add 
# .option("partitionColumn", "some_id").option("lowerBound", "0").option("upperBound", "90000000").option("numPartitions", "4")
#df = spark.read.jdbc(url=url, table="dbo.customerdist_1910a", properties=properties)
#df = spark.read.jdbc(url=url, table="dbo.branch", properties=properties)
#df = spark.read.jdbc(url=url, table="dbo.product", properties=properties)
#df = spark.read.jdbc(url=url, table="dbo.prodcat", properties=properties)
#df = spark.read.jdbc(url=url, table="dbo.discount_b", properties=properties)
#df = spark.read.jdbc(url=url, table="dbo.discount_potTunai", properties=properties)
#df = spark.read.jdbc(url=url, table="dbo.sls_chlprn", properties=properties)
#df = spark.read.jdbc(url=url, table="dbo.slsmanexc", properties=properties)
#df = spark.read.jdbc(url=url, table="dbo.slsexc_chlprn6", properties=properties)
#df = spark.read.jdbc(url=url, table="dbo.slsman", properties=properties)
#df = spark.read.jdbc(url=url, table="dbo.dw_ilo_18", properties=properties)
#df = spark.read.jdbc(url=url, table="dbo.dw_ilo_oth", properties=properties)
#df = spark.read.jdbc(url=url, table="dbo.sttwk_2022a", properties=properties)
end_time = time.time()
print(f"Total time taken for load: {end_time - start_time:.2f} seconds")


Step 1: Reading from SQL Server View...
Total time taken for load: 1.11 seconds


In [5]:
# 3. Save to Parquet (This is the fastest 'bridge' to DuckDB)
start_time = time.time()
#to handle too few records
# Disable the statistics tracker that is causing the 'None.get' error
#below is no need for spark 4.0, but needed for transfering small table in spark 3.5.6
'''
spark.conf.set("spark.sql.sources.commitProtocolClass", 
               "org.apache.spark.sql.execution.datasources.SQLHadoopMapReduceCommitProtocol")
spark.conf.set("spark.sql.parquet.fs.optimized.committer.optimization-enabled", "false")
'''
print("Step 2: Exporting to Parquet (This saves your RAM)...")
start_time = time.time()

# Write without coalesce(1) to keep it fast
print("Exporting in parallel...")
df.write.mode("overwrite").option("compression", "snappy").parquet(TEMP_PARQUET)
# .coalesce(1) ensures one single file is written for small tables
# and prevents the metrics 'None.get' error
#df.coalesce(1).write.mode("overwrite").parquet(TEMP_PARQUET)

end_time = time.time()
print(f"Total time taken for parquet: {end_time - start_time:.2f} seconds")

Step 2: Exporting to Parquet (This saves your RAM)...
Exporting in parallel...
Total time taken for parquet: 173.89 seconds


In [ ]:
# do this in duckdb command prompt (CLI with D prompt)
CREATE OR REPLACE TABLE customerdist_1910a AS 
SELECT * FROM read_parquet('C:\duckDB\bridge_customerdist\*.parquet');

In [12]:
import duckdb
import os
import time

# 1. Connect using the exact path you provided
DB_PATH = r'C:\DuckDB\AdventureSales.duckdb'
con = duckdb.connect(DB_PATH)

# 2. Automatically find bridge folders (Case-Insensitive)
base_path = r'C:\DuckDB'
# This handles 'bridge', 'Bridge', or 'BRIDGE'
bridge_folders = [f for f in os.listdir(base_path) if f.lower().startswith('bridge')]

print(f"Found {len(bridge_folders)} folders in {base_path}. Syncing now...")

for folder in bridge_folders:
    # Get the table name (e.g., bridge_branch -> branch)
    raw_name = folder.lower().replace('bridge_', '')
    
    # Mapping fix for your specific customerdist table
    table_name = 'customerdist_1910a' if 'customerdist' in raw_name else raw_name
    
    # Use forward slashes and the absolute folder path
    full_path = f"C:/DuckDB/{folder}/**/*.parquet"
    
    try:
        con.execute(f"CREATE OR REPLACE TABLE {table_name} AS SELECT * FROM read_parquet('{full_path}')")
        print(f"  [OK] Synchronized: {table_name}")
    except Exception as e:
        print(f"  [!] Failed to sync {table_name}: {e}")

# 3. VERIFY
print("\nFinal Verification - Tables in AdventureSales:")
con.sql("SHOW TABLES").show()

# 4. RUN THE REPORT (Only if tables exist)
print("Executing 88M row report query...")
start = time.time()

# Note: DuckDB's 'GROUP BY ALL' is a life-saver here!
con.execute(query) # Using the 'query' variable we defined earlier

duration = time.time() - start
print(f"\n==========================================")
print(f"REPORT FINISHED IN {duration:.2f} SECONDS!")
print(f"==========================================")

con.close()


Found 0 folders in C:\DuckDB. Syncing now...

Final Verification - Tables in AdventureSales:
┌─────────┐
│  name   │
│ varchar │
├─────────┤
│ sttwk   │
└─────────┘

Executing 88M row report query...


CatalogException: Catalog Error: Table with name customerdist_1910a does not exist!
Did you mean "sqlite_master"?

LINE 26: LEFT JOIN customerdist_1910a c ON a.custd_id = c.kode_iap
                   ^